## Assignment 2: $k$ Nearest Neighbor

### Do any four.

**Q1.** Please answer the following questions.
1. What is the difference between regression and classification?
2. What is a confusion table/matrix? What does it help us understand about a model's performance? 
3. What is Accuracy? Why might it not be entirely sufficient to evaluate a classifer's predictive performance?
4. What does the root mean squared error quantify about a particular model?
5. What are overfitting and underfitting? 
6. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or RMSE on the test set, improve model performance?
7. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

Q1 answers

1. Regression predicts a numeric outcome, while classification predicts a category or class label.

2. A confusion matrix compares actual classes to predicted classes. It shows which classes are predicted correctly and which classes are being mixed up.

3. Accuracy is the proportion of predictions that are correct. It can be misleading when classes are imbalanced because a model can look good overall while performing poorly on a rare but important class.

4. RMSE measures the typical size of prediction errors for a regression model, with larger mistakes penalized more heavily because the errors are squared.

5. Overfitting occurs when a model learns the training data too well, including noise and random patterns, leading to poor performance on new data. Underfitting occurs when a model is too simple to capture the underlying patterns in the data, resulting in poor performance on both training and test data.

6. Splitting data into training and testing sets allows us to evaluate model performance on unseen data. We use the training set to fit the model and the test set to select the best k value. This prevents overfitting because we're choosing k based on performance on data the model hasn't seen during training, which better reflects how it will perform on new data.

7. Predicting class labels gives a single definitive answer, which is simple and actionable but doesn't convey uncertainty. Predicting probabilities shows the model's confidence across all classes, which is more informative for decision-making (especially when costs of different errors vary) but requires the user to set a threshold or interpret the probabilities.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the relationships between the features (e.g. scatterplots, describe tables).
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for your best model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

In [ ]:
# Necessary helper code to answer all questions

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

#seaborn theme for consistent plot styling
sns.set_theme(style="whitegrid")

# constants
DATA_DIR = "./data"
RANDOM_STATE = 42

#read CSV files with multiple encoding attempts
def read_csv_robust(path, **kwargs):
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, encoding_errors="replace", **kwargs)
        except UnicodeDecodeError:
            pass
    return pd.read_csv(path, encoding="latin1", encoding_errors="replace", **kwargs)

#normalize training and test data using MinMaxScaler
def minmax_train_test(X_train, X_test):
    scaler = MinMaxScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_scaled, X_test_scaled, scaler

#compute MSE and RMSE for regression models
def regression_summary(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    return mse, rmse

In [ ]:
# Q2.1: Load the land mines data
land_mines = read_csv_robust(f"{DATA_DIR}/land_mines.csv")

#shape, preview data, and look for missing values
land_mines.shape, land_mines.head(), land_mines.isna().sum()

In [ ]:
# Q2.1: Perform EDA on the land mines data
land_mines["mine_type"].value_counts().sort_index(), land_mines.describe()

#plot
sns.pairplot(land_mines, hue="mine_type", diag_kind="hist")
plt.show()

In [ ]:
# Q2.2 & Q2.3: Prepare features and target, then split into train/val/test
X = land_mines[["voltage", "height", "soil"]]
y = land_mines["mine_type"]

#split 50% train, 50% temp (for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.5, random_state=RANDOM_STATE, stratify=y
)

#split temp into 50% validation, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

# Test multiple k values using validation set to find the best one
candidate_k = [1, 3, 5, 7, 9, 11, 15, 21]
q2_results = []
for k in candidate_k:
    # Train knn classifier with k neighbors
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    q2_results.append({"k": k, "val_accuracy": accuracy_score(y_val, pred)})

# Sort by accuracy (descending) and k (ascending) to find best k
q2_results = pd.DataFrame(q2_results).sort_values(["val_accuracy", "k"], ascending=[False, True])
q2_results

In [ ]:
# Q2.4: Train final model with best k and evaluate
# best k value from results
best_k_q2 = int(q2_results.iloc[0]["k"])
# Train final model with optimal k
q2_model = KNeighborsClassifier(n_neighbors=best_k_q2)
q2_model.fit(X_train, y_train)

# predictions on test set
q2_pred = q2_model.predict(X_test)

#confusion matrix as a DataFrame for better readability
q2_conf = pd.DataFrame(
    confusion_matrix(y_test, q2_pred, labels=sorted(y.unique())),
    index=[f"actual_{c}" for c in sorted(y.unique())],
    columns=[f"pred_{c}" for c in sorted(y.unique())],
)

print(f"Best k: {best_k_q2}")
print(f"Accuracy: {accuracy_score(y_test, q2_pred):.4f}")
print("\nConfusion matrix:")
display(q2_conf)

print("\nThe model achieves", f"{accuracy_score(y_test, q2_pred):.1%}", "accuracy on the test set.")
print("Looking at the confusion matrix, we can see which mine types are easier or harder to classify correctly.")

Q2.3: I selected k by comparing validation-set accuracy across several candidate values and choosing the one with the highest accuracy. This avoids data leakage by keeping the test set separate for final evaluation only.

Q2.4: The accuracy and confusion matrix are shown above. The confusion matrix shows which mine types are easiest and hardest to distinguish.

Q2.5: In practice, I would use this model as a screening tool rather than as a final decision-maker, because even a fairly accurate model can still make dangerous class-level mistakes.

**Q3.** This question is a case study for $k$ nearest neighbor regression, using the `USA_cars_datasets.csv` data.

The target variable `y` is `price` and the features are `year` and `mileage`.

1. Load the `./data/USA_cars_datasets.csv`. Keep the following variables and drop the rest: `price`, `year`, `mileage`. Are there any `NA`'s to handle? Look at the head and dimensions of the data.
2. Maxmin normalize `year` and `mileage`.
3. Split the sample into ~80% for training and ~20% for hyper-parameter selection and evaluation.
4. Use the $k$-NN algorithm and the training data to predict `price` using `year` and `mileage` for the test set for $k=3,10,25,50,100,300$. For each value of $k$, compute the mean squared error and print a scatterplot showing the test value plotted against the predicted value. What patterns do you notice as you increase $k$?
5. Determine the optimal $k$ for these data.
6. Describe what happened in the plots of predicted versus actual prices as $k$ varied, taking your answer into part 6 into account. (Hint: Use the words "underfitting" and "overfitting".)

In [ ]:
# Q3.1: cars data and keep only price, year, and mileage
cars = read_csv_robust(f"{DATA_DIR}/USA_cars_datasets.csv")
cars = cars[["price", "year", "mileage"]].copy()
# Convert all columns to numeric, coercing errors to NaN
for c in cars.columns:
    cars[c] = pd.to_numeric(cars[c], errors="coerce")

cars.shape, cars.head(), cars.isna().sum()

In [ ]:
# Q3.2-3.5: Prepare data, normalize, split, and test multiple k values
X = cars[["year", "mileage"]]
y = cars["price"]

# Q3.3: 80/20 for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# Q3.2: Normalize year and mileage using MinMax scaling
X_train_s, X_test_s, scaler_cars = minmax_train_test(X_train, X_test)

# Q3.4: Test k values and create scatterplots
k_grid_q3 = [3, 10, 25, 50, 100, 300]
q3_results = []

# Create 2x3 grid of subplots for the 6 k values
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

# For each k value, train model and create scatterplot
for ax, k in zip(axes, k_grid_q3):
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    mse, rmse = regression_summary(y_test, pred)
    q3_results.append({"k": k, "mse": mse, "rmse": rmse})

    #scatterplot of actual vs predicted prices
    ax.scatter(y_test, pred, alpha=0.35)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, color="red", linewidth=1)
    ax.set_title(f"k={k}, MSE={mse:.0f}")
    ax.set_xlabel("Actual price")
    ax.set_ylabel("Predicted price")

plt.tight_layout()
plt.show()

q3_results_df = pd.DataFrame(q3_results).sort_values("mse")
print("\nMSE results for different k values:")
display(q3_results_df)

print(f"\nQ3.4: As k increases, the predictions become smoother and less responsive to local patterns.")
print(f"Q3.5: The optimal k is {int(q3_results_df.iloc[0]['k'])} with MSE = {q3_results_df.iloc[0]['mse']:.2f}")

Q3.6: Very small k can overfit by chasing local noise in the training data, while very large k can underfit by averaging over too many different cars and pulling predictions toward the center of the price distribution. The optimal k balances these two extremes.

**Q5.** This is a case study on $k$ nearest neighbor classification, using the `animals.csv` data.

The data consist of a label, `class`, taking integer values 1 to 7, the name of the species, `animal`, and 16 characteristics of the animal, including `hair`, `feathers`, `milk`, `eggs`, `airborne`, and so on. 

1. Load the data. For each of the seven class labels, print the values in the class and get a sense of what is included in that group. Perform some other EDA: How big are the classes? How much variation is there in each of the features/covariates? Which variables do you think will best predict which class?
2. Split the data 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be. Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Using all of the variables, build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion matrix for the optimal model, comparing predicted and actual class label on the test set. How accurate it is? Can you interpret why mistakes are made across groups?
5. Use only `milk`, `aquatic`, and `airborne` to train a new $k$-NN classifier. Print your confusion table. Mine does not predict all of the classes, only a subset of them. To see the underlying proportions/probabilities, use `model.predict_proba(X_test.values)` to predict probabilities rather than labels for your `X_test` test data for your fitted `model`. Are all of the classes represented? Explain your results.

In [ ]:
# Q5.1: Load the animals data
animals = read_csv_robust(f"{DATA_DIR}/zoo.csv")

animals.shape, animals.head(), animals["class"].value_counts().sort_index()

In [ ]:
# Q5.1: Print animals in each class and perform EDA
print("Q5.1: Printing animals in each class:")
#loop through each class and print the animals in that class
for class_id in sorted(animals["class"].unique()):
    print(f"\nClass {class_id}:")
    print(animals.loc[animals["class"] == class_id, "animal"].tolist())

print("\n\nClass sizes:")
display(animals["class"].value_counts().sort_index())

print("\n\nFeature variation:")
display(animals.drop(columns=["animal"]).describe())

In [ ]:
# Q5.2 & Q5.3: Prepare features, split data, and find optimal k
# feature columns except animal name and class label
feature_cols = [c for c in animals.columns if c not in ["animal", "class"]]
X = animals[feature_cols]
y = animals["class"]

#split: 50% train, 50% temp (for validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.5, random_state=RANDOM_STATE, stratify=y
)

#split temp into 50% validation, 50% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

# Q5.3: Test multiple k values using validation set to find optimal one
k_grid_q5 = [1, 3, 5, 7, 9]
q5_results = []
for k in k_grid_q5:
    # Train kNN classifier with k neighbors
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    # Predict on validation set (not test set)
    pred = model.predict(X_val)
    # Store k and validation accuracy
    q5_results.append({"k": k, "val_accuracy": accuracy_score(y_val, pred)})

# Sort by accuracy (descending) and k (ascending) to find best k
q5_results_df = pd.DataFrame(q5_results).sort_values(["val_accuracy", "k"], ascending=[False, True])
print("Q5.3: k selection results:")
display(q5_results_df)
print(f"\nI selected k by comparing validation-set accuracy and chose k = {int(q5_results_df.iloc[0]['k'])}")

In [ ]:
# Q5.4: Train final model with best k and create confusion matrix
#best k value from results
best_k_q5 = int(q5_results_df.iloc[0]["k"])
# Train final model with optimal k
q5_model = KNeighborsClassifier(n_neighbors=best_k_q5)
q5_model.fit(X_train, y_train)
# Make predictions on test set
q5_pred = q5_model.predict(X_test)

#confusion matrix as a DataFrame for better readability
q5_conf = pd.DataFrame(
    confusion_matrix(y_test, q5_pred, labels=sorted(y.unique())),
    index=[f"actual_{c}" for c in sorted(y.unique())],
    columns=[f"pred_{c}" for c in sorted(y.unique())],
)

print(f"Q5.4: Confusion matrix for optimal model (k={best_k_q5}):")
display(q5_conf)
print(f"\nAccuracy: {accuracy_score(y_test, q5_pred):.4f}")
print("\nThe confusion matrix shows which classes are predicted accurately and where mistakes occur.")

In [ ]:
# Q5.5: Train model with only 3 features and examine probabilities
# only milk, aquatic, and airborne features
X_small = animals[["milk", "aquatic", "airborne"]]
# Split data with same parameters as before
X_train_s, X_temp_s, y_train_s, y_temp_s = train_test_split(
    X_small, y, test_size=0.5, random_state=RANDOM_STATE, stratify=y
)
X_val_s, X_test_s, y_val_s, y_test_s = train_test_split(
    X_temp_s, y_temp_s, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_s
)

# Train kNN classifier with optimal k from before
q5_small_model = KNeighborsClassifier(n_neighbors=best_k_q5)
q5_small_model.fit(X_train_s, y_train_s)
# Make predictions on test set
q5_small_pred = q5_small_model.predict(X_test_s)
# Get probability predictions for each class
q5_small_proba = pd.DataFrame(q5_small_model.predict_proba(X_test_s), columns=q5_small_model.classes_)

# confusion matrix
q5_small_conf = pd.DataFrame(
    confusion_matrix(y_test_s, q5_small_pred, labels=sorted(y.unique())),
    index=[f"actual_{c}" for c in sorted(y.unique())],
    columns=[f"pred_{c}" for c in sorted(y.unique())],
)

print("Q5.5: Confusion matrix using only milk, aquatic, and airborne:")
display(q5_small_conf)

print("\nPredicted probabilities (first 5 test observations):")
display(q5_small_proba.head())

print(f"\nClasses represented in predictions: {sorted(q5_small_model.classes_)}")
print("\nWith only 3 features, several classes overlap and the model may not predict all 7 classes.")
print("The probability output shows which classes receive nonzero support even when not predicted as the final label.")